# Resume Quality + Career Recommendation System (Stricter Scoring + Optional Dataset)

This version is cleaned into smaller cells, keeps backend-ready saving, **uses the optional `netsol/resume-score-details` dataset when available**, and makes scoring **slightly less generous**.

## Saved artifacts
After training, this notebook saves these files inside `artifacts/`:

- `quality_bundle.joblib`
- `career_bundle.joblib`
- `job_fit_bundle.joblib` *(optional)*
- `training_summary.json`

These are the files you should load in your backend.


## 0) Install packages

Run this once.  
If `datasets` fails in your environment, the notebook still works without the optional dataset.


In [1]:
!pip install --upgrade pip
!pip install pymupdf python-docx joblib numpy pandas scikit-learn ipython datasets huggingface_hub

  Using cached pip-26.0.1-py3-none-any.whl.metadata (4.7 kB)
Using cached pip-26.0.1-py3-none-any.whl (1.8 MB)


ERROR: To modify pip, please run the following command:
C:\Users\LENOVO\anaconda3\python.exe -m pip install --upgrade pip


## 1) Imports

In [2]:
from __future__ import annotations

import json
import random
import re
import urllib.request
import warnings
from pathlib import Path
from typing import Dict, List, Tuple

import fitz
import joblib
import numpy as np
import pandas as pd
from docx import Document
from IPython.display import display
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, top_k_accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

warnings.filterwarnings("ignore")

## 2) Project paths and constants

In [3]:
RANDOM_STATE = 42

DATA_DIR = Path("data")
ARTIFACT_DIR = Path("artifacts")

DATA_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

DEFAULT_DATA_URLS = {
    "career": "https://huggingface.co/datasets/ahmedheakl/resume-atlas/resolve/main/train.csv?download=true",
    "ats": "https://huggingface.co/datasets/0xnbk/resume-ats-score-v1-en/resolve/main/train.csv?download=true",
}

OPTIONAL_DATASET_NAME = "netsol/resume-score-details"

QUALITY_CLASS_ORDER = ["poor", "fair", "good", "excellent"]

# Slightly stricter than before
QUALITY_SCORE_ANCHORS = {
    "poor": 28.0,
    "fair": 50.0,
    "good": 72.0,
    "excellent": 88.0,
}

QUALITY_RATING_THRESHOLDS = {
    "poor_max": 44.0,
    "fair_max": 61.0,
    "good_max": 79.0,
}

## 3) Regex patterns

In [4]:
SECTION_PATTERNS = {
    "summary": [r"\bsummary\b", r"\bprofile\b", r"\bobjective\b"],
    "experience": [r"\bexperience\b", r"\bwork history\b", r"\bemployment\b"],
    "education": [r"\beducation\b", r"\bacademic\b"],
    "skills": [r"\bskills\b", r"\btechnical skills\b", r"\bcore competencies\b"],
    "projects": [r"\bprojects\b", r"\bproject experience\b"],
    "certifications": [r"\bcertifications\b", r"\blicenses\b", r"\bcertificates\b"],
}

EMAIL_RE = re.compile(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}")
PHONE_RE = re.compile(r"(?:\+?\d[\d\-\s().]{7,}\d)")
YEAR_RE = re.compile(r"(?:19|20)\d{2}")

## 4) Small utility functions

In [5]:
def maybe_download(url: str, path: Path) -> bool:
    if path.exists():
        return True
    try:
        urllib.request.urlretrieve(url, path)
        return True
    except Exception as exc:
        print(f"Download failed for {path.name}: {exc}")
        return False

def safe_text(x) -> str:
    if x is None:
        return ""
    return str(x).strip()

def clip_score(x: float, lo: float = 0.0, hi: float = 100.0) -> float:
    return float(max(lo, min(hi, x)))

def score_to_band(score: float) -> str:
    score = float(score)
    if score <= QUALITY_RATING_THRESHOLDS["poor_max"]:
        return "poor"
    if score <= QUALITY_RATING_THRESHOLDS["fair_max"]:
        return "fair"
    if score <= QUALITY_RATING_THRESHOLDS["good_max"]:
        return "good"
    return "excellent"

def mean_or_none(values: List[float]):
    values = [float(v) for v in values if v is not None and not pd.isna(v)]
    return float(np.mean(values)) if values else None

## 5) Resume file readers

In [6]:
def read_pdf(path: str) -> str:
    doc = fitz.open(path)
    pages = [page.get_text("text") for page in doc]
    return "\n".join(pages).strip()

def read_docx(path: str) -> str:
    doc = Document(path)
    return "\n".join(p.text for p in doc.paragraphs).strip()

def read_txt(path: str) -> str:
    return Path(path).read_text(encoding="utf-8", errors="ignore").strip()

def read_resume(path: str) -> str:
    suffix = Path(path).suffix.lower()
    if suffix == ".pdf":
        return read_pdf(path)
    if suffix == ".docx":
        return read_docx(path)
    if suffix in {".txt", ".md"}:
        return read_txt(path)
    raise ValueError(f"Unsupported file type: {suffix}")

## 6) Structural feature extraction

In [7]:
def structural_features(text: str) -> Dict[str, float]:
    text = safe_text(text)
    lowered = text.lower()

    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    words = re.findall(r"\b\w+\b", text)
    bullets = [ln for ln in lines if re.match(r"^(?:[-*•]|\d+[.)])\s+", ln)]
    section_hits = 0
    for patterns in SECTION_PATTERNS.values():
        if any(re.search(p, lowered, flags=re.I) for p in patterns):
            section_hits += 1

    quantified = 0
    for ln in bullets:
        if re.search(r"\d|%|million|billion|kpi|revenue|cost|growth|improv", ln, flags=re.I):
            quantified += 1

    return {
        "word_count": float(len(words)),
        "line_count": float(len(lines)),
        "bullet_count": float(len(bullets)),
        "section_coverage": float(section_hits),
        "email_count": float(len(EMAIL_RE.findall(text))),
        "phone_count": float(len(PHONE_RE.findall(text))),
        "year_mentions": float(len(YEAR_RE.findall(text))),
        "avg_line_length": float(np.mean([len(ln.split()) for ln in lines])) if lines else 0.0,
        "quantified_bullet_ratio": float(quantified / max(1, len(bullets))),
    }

def structural_frame(texts: List[str]) -> pd.DataFrame:
    return pd.DataFrame([structural_features(t) for t in texts])

## 7) ATS text parsing helpers

In [8]:
def extract_resume_from_ats_text(text: str) -> str:
    text = safe_text(text)
    markers = [
        "resume:",
        "candidate resume:",
        "candidate profile:",
    ]
    cut = text
    for marker in markers:
        idx = text.lower().find(marker)
        if idx != -1:
            cut = text[idx + len(marker):]
            break

    jd_markers = ["job description:", "jd:", "target role:"]
    end_idx = len(cut)
    for marker in jd_markers:
        idx = cut.lower().find(marker)
        if idx != -1:
            end_idx = min(end_idx, idx)
    return cut[:end_idx].strip()

def extract_job_from_ats_text(text: str) -> str:
    text = safe_text(text)
    jd_markers = ["job description:", "jd:", "target role:"]
    for marker in jd_markers:
        idx = text.lower().find(marker)
        if idx != -1:
            return text[idx + len(marker):].strip()
    return 

## 8) Load ResumeAtlas and ATS datasets

In [9]:
def load_base_datasets() -> Tuple[pd.DataFrame, pd.DataFrame, Dict[str, str]]:
    career_path = DATA_DIR / "resume_atlas.csv"
    ats_path = DATA_DIR / "ats_train.csv"

    career_ok = maybe_download(DEFAULT_DATA_URLS["career"], career_path)
    ats_ok = maybe_download(DEFAULT_DATA_URLS["ats"], ats_path)

    if not career_ok:
        raise RuntimeError("ResumeAtlas could not be loaded. Please place data/resume_atlas.csv manually.")

    career_df = pd.read_csv(career_path)
    career_df = career_df.rename(columns={"Text": "resume_text", "Category": "category"})
    career_df = career_df.dropna(subset=["resume_text", "category"]).copy()
    career_df["resume_text"] = career_df["resume_text"].astype(str)

    if ats_ok:
        ats_df = pd.read_csv(ats_path)
        ats_df = ats_df.dropna(subset=["text", "ats_score"]).copy()
        ats_df["resume_text"] = ats_df["text"].map(extract_resume_from_ats_text)
        ats_df["job_text"] = ats_df["text"].map(extract_job_from_ats_text)
    else:
        ats_df = pd.DataFrame(columns=["text", "ats_score", "resume_text", "job_text"])

    source_info = {
        "career": "real",
        "ats": "real" if len(ats_df) else "missing_optional",
    }
    return career_df, ats_df, source_info

## 9) Optional score-details dataset

This tries two routes:

1. local file/folder inside `data/`
2. Hugging Face dataset loader for `netsol/resume-score-details`

If neither works, the notebook still trains normally.


In [10]:
def parse_optional_record(rec: Dict) -> Dict[str, object] | None:
    try:
        inp = rec.get("input", {}) or {}
        out = rec.get("output", {}) or {}
        details = rec.get("details", {}) or {}
        scores = (out.get("scores", {}) or {})
        agg = scores.get("aggregated_scores", {}) or {}

        resume_text = safe_text(inp.get("resume"))
        job_description = safe_text(inp.get("job_description"))

        macro = agg.get("macro_scores")
        micro = agg.get("micro_scores")
        valid = out.get("valid_resume_and_jd", True)

        if not resume_text:
            return None

        raw_5_scale = mean_or_none([macro, micro])
        weak_score = None if raw_5_scale is None else clip_score(raw_5_scale * 20.0)

        return {
            "resume_text": resume_text,
            "job_description": job_description,
            "weak_score": weak_score,
            "valid_pair": bool(valid),
            "detail_skill_count": float(len(details.get("skills", []) or [])),
        }
    except Exception:
        return None

def load_optional_score_details_dataset() -> pd.DataFrame:
    # A) local JSON files placed under data/resume_score_details/
    local_dir = DATA_DIR / "resume_score_details"
    local_jsonl = DATA_DIR / "resume_score_details.jsonl"
    local_csv = DATA_DIR / "resume_score_details.csv"

    rows = []

    if local_csv.exists():
        temp = pd.read_csv(local_csv)
        for _, r in temp.iterrows():
            rows.append({
                "resume_text": safe_text(r.get("resume_text")),
                "job_description": safe_text(r.get("job_description")),
                "weak_score": pd.to_numeric(r.get("weak_score"), errors="coerce"),
                "valid_pair": bool(r.get("valid_pair", True)),
                "detail_skill_count": pd.to_numeric(r.get("detail_skill_count", 0), errors="coerce"),
            })

    elif local_jsonl.exists():
        with open(local_jsonl, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                item = parse_optional_record(json.loads(line))
                if item is not None:
                    rows.append(item)

    elif local_dir.exists():
        for fp in sorted(local_dir.glob("*.json")):
            try:
                item = parse_optional_record(json.loads(fp.read_text(encoding="utf-8")))
                if item is not None:
                    rows.append(item)
            except Exception:
                pass

    if rows:
        df = pd.DataFrame(rows)
        df = df.dropna(subset=["resume_text"]).copy()
        df["resume_text"] = df["resume_text"].astype(str)
        return df

    # B) fallback: try Hugging Face datasets library
    try:
        from datasets import load_dataset
        ds = load_dataset(OPTIONAL_DATASET_NAME, split="train")
        rows = []
        for rec in ds:
            item = parse_optional_record(rec)
            if item is not None:
                rows.append(item)
        if rows:
            df = pd.DataFrame(rows)
            df = df.dropna(subset=["resume_text"]).copy()
            df["resume_text"] = df["resume_text"].astype(str)
            return df
    except Exception as exc:
        print(f"Optional dataset not loaded: {exc}")

    return pd.DataFrame(columns=["resume_text", "job_description", "weak_score", "valid_pair", "detail_skill_count"])

## 10) Build synthetic quality training data

In [11]:
def degrade_resume(text: str, severity: int, rng: random.Random) -> str:
    text = safe_text(text)

    if severity == 1:
        text = EMAIL_RE.sub("", text)
        text = PHONE_RE.sub("", text)
        text = re.sub(r"\blinkedin\S*", "", text, flags=re.I)
        kept = []
        for tok in text.split():
            if re.search(r"\d|%", tok) and rng.random() < 0.15:
                continue
            kept.append(tok)
        text = " ".join(kept)

    elif severity == 2:
        text = EMAIL_RE.sub("", text)
        text = PHONE_RE.sub("", text)
        text = re.sub(r"\b(projects?|certifications?|awards)\b", "", text, flags=re.I)
        text = text.replace("•", " ").replace("- ", " ")
        tokens = text.split()
        cutoff = max(80, int(len(tokens) * 0.62))
        text = " ".join(tokens[:cutoff])

    elif severity == 3:
        tokens = text.split()
        tokens = tokens[: max(40, int(len(tokens) * 0.30))]
        rng.shuffle(tokens)
        text = " ".join(tokens).lower()
        text = re.sub(
            r"\b(summary|profile|objective|experience|education|skills|projects|certifications)\b",
            "",
            text,
            flags=re.I,
        )

    return re.sub(r"\s+", " ", text).strip()

def make_synthetic_quality_frame(resumes: List[str], max_samples: int = 3000, seed: int = RANDOM_STATE) -> pd.DataFrame:
    rng = random.Random(seed)
    resumes = [safe_text(x) for x in resumes if len(safe_text(x).split()) >= 60]

    if len(resumes) > max_samples:
        idx = rng.sample(range(len(resumes)), max_samples)
        resumes = [resumes[i] for i in idx]

    rows = []
    for text in resumes:
        rows.append({"resume_text": text, "quality_label": "excellent", "source": "synthetic"})
        rows.append({"resume_text": degrade_resume(text, 1, rng), "quality_label": "good", "source": "synthetic"})
        rows.append({"resume_text": degrade_resume(text, 2, rng), "quality_label": "fair", "source": "synthetic"})
        rows.append({"resume_text": degrade_resume(text, 3, rng), "quality_label": "poor", "source": "synthetic"})
    return pd.DataFrame(rows)

## 11) Convert optional weak-score dataset into extra quality labels

In [12]:
def make_optional_quality_frame(optional_df: pd.DataFrame) -> pd.DataFrame:
    if optional_df is None or len(optional_df) == 0:
        return pd.DataFrame(columns=["resume_text", "quality_label", "source"])

    df = optional_df.copy()
    df = df.dropna(subset=["resume_text"]).copy()
    df["resume_text"] = df["resume_text"].astype(str)
    df = df[df["resume_text"].str.split().str.len() >= 40].copy()

    score = pd.to_numeric(df["weak_score"], errors="coerce")
    bands = []
    for s, valid in zip(score, df.get("valid_pair", pd.Series([True] * len(df)))):
        if not valid:
            bands.append("poor")
        elif pd.isna(s):
            bands.append(None)
        elif s < 45:
            bands.append("poor")
        elif s < 62:
            bands.append("fair")
        elif s < 80:
            bands.append("good")
        else:
            bands.append("excellent")

    df["quality_label"] = bands
    df = df.dropna(subset=["quality_label"]).copy()
    df["source"] = "optional_score_details"
    return df[["resume_text", "quality_label", "source"]]

## 12) Train stricter intrinsic quality model

In [13]:
def fit_quality_model(
    resumes: List[str],
    optional_df: pd.DataFrame | None = None,
    max_samples: int = 3000,
) -> Dict[str, object]:
    synthetic_df = make_synthetic_quality_frame(resumes, max_samples=max_samples)
    optional_quality_df = make_optional_quality_frame(optional_df)

    quality_df = synthetic_df.copy()
    if len(optional_quality_df):
        quality_df = pd.concat([quality_df, optional_quality_df], ignore_index=True)

    X_train, X_test, y_train, y_test = train_test_split(
        quality_df["resume_text"],
        quality_df["quality_label"],
        test_size=0.2,
        random_state=RANDOM_STATE,
        stratify=quality_df["quality_label"],
    )

    vectorizer = TfidfVectorizer(
        lowercase=True,
        strip_accents="unicode",
        stop_words="english",
        ngram_range=(1, 2),
        max_features=12000,
        min_df=2,
    )

    Xtr_text = vectorizer.fit_transform(X_train.tolist())
    Xte_text = vectorizer.transform(X_test.tolist())

    train_struct = structural_frame(X_train.tolist())
    test_struct = structural_frame(X_test.tolist())

    imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()

    Xtr_struct = scaler.fit_transform(imputer.fit_transform(train_struct))
    Xte_struct = scaler.transform(imputer.transform(test_struct))

    Xtr = hstack([Xtr_text, csr_matrix(Xtr_struct)])
    Xte = hstack([Xte_text, csr_matrix(Xte_struct)])

    model = LogisticRegression(
        max_iter=1200,
        solver="lbfgs",
        random_state=RANDOM_STATE,
    )
    model.fit(Xtr, y_train)

    pred = model.predict(Xte)
    prob = model.predict_proba(Xte)

    metrics = {
        "accuracy": float(accuracy_score(y_test, pred)),
        "top_2_accuracy": float(top_k_accuracy_score(y_test, prob, k=2, labels=model.classes_)),
        "train_rows": int(len(quality_df)),
        "synthetic_rows": int(len(synthetic_df)),
        "optional_rows_used": int(len(optional_quality_df)),
    }

    feature_percentiles = train_struct.quantile([0.25, 0.5, 0.75]).to_dict()

    return {
        "task": "intrinsic_resume_quality",
        "vectorizer": vectorizer,
        "imputer": imputer,
        "scaler": scaler,
        "model": model,
        "metrics": metrics,
        "feature_percentiles": feature_percentiles,
        "score_anchors": QUALITY_SCORE_ANCHORS,
        "rating_thresholds": QUALITY_RATING_THRESHOLDS,
    }

## 13) Quality feedback

In [14]:
def quality_feedback(text: str, quality_bundle: Dict[str, object]) -> List[str]:
    feat = structural_features(text)
    fp = quality_bundle["feature_percentiles"]
    notes: List[str] = []

    def q25(name: str) -> float:
        return float(fp.get(name, {}).get(0.25, 0.0))

    if feat["section_coverage"] < max(4.0, q25("section_coverage")):
        notes.append("Add or clarify standard sections such as Summary, Experience, Education, Skills, and Projects.")
    if feat["quantified_bullet_ratio"] < max(0.15, q25("quantified_bullet_ratio")):
        notes.append("Use more quantified achievements with numbers, percentages, budget, scale, or outcome.")
    if feat["bullet_count"] < max(6.0, q25("bullet_count")):
        notes.append("Add more concise bullet points so recruiters can scan achievements faster.")
    if feat["word_count"] < max(250.0, q25("word_count")):
        notes.append("The resume may be too thin. Add stronger detail for experience, projects, and accomplishments.")
    if feat["email_count"] < 1 or feat["phone_count"] < 1:
        notes.append("Include clear contact details such as email and phone number.")
    if not notes:
        notes.append("Overall structure looks strong. Next improvements should focus on sharper achievement wording and better role targeting.")
    return notes

## 14) Slightly stricter score calibration

In [15]:
def score_penalty_from_structure(feat: Dict[str, float]) -> float:
    penalty = 0.0

    if feat["section_coverage"] < 4:
        penalty += (4 - feat["section_coverage"]) * 3.0

    if feat["email_count"] < 1:
        penalty += 4.0

    if feat["phone_count"] < 1:
        penalty += 4.0

    if feat["quantified_bullet_ratio"] < 0.15:
        penalty += min(8.0, (0.15 - feat["quantified_bullet_ratio"]) * 30.0)

    if feat["bullet_count"] < 6:
        penalty += (6 - feat["bullet_count"]) * 0.8

    if feat["word_count"] < 250:
        penalty += min(8.0, (250 - feat["word_count"]) / 25.0)

    if feat["word_count"] > 900:
        penalty += min(4.0, (feat["word_count"] - 900) / 100.0)

    return float(max(0.0, penalty))

def score_resume_quality(text: str, quality_bundle: Dict[str, object]) -> Dict[str, object]:
    X_text = quality_bundle["vectorizer"].transform([text])
    X_struct = structural_frame([text])
    X_struct = quality_bundle["scaler"].transform(quality_bundle["imputer"].transform(X_struct))
    X = hstack([X_text, csr_matrix(X_struct)])

    classes = list(quality_bundle["model"].classes_)
    prob = quality_bundle["model"].predict_proba(X)[0]
    prob_map = {label: float(prob[i]) for i, label in enumerate(classes)}

    raw_score = 0.0
    for label, anchor in quality_bundle["score_anchors"].items():
        raw_score += prob_map.get(label, 0.0) * anchor

    feat = structural_features(text)
    penalty = score_penalty_from_structure(feat)
    final_score = clip_score(raw_score - penalty)

    rating = score_to_band(final_score)

    return {
        "raw_score": round(float(raw_score), 1),
        "penalty": round(float(penalty), 1),
        "score": round(float(final_score), 1),
        "rating": rating,
        "probabilities": prob_map,
        "features": feat,
        "feedback": quality_feedback(text, quality_bundle),
    }

## 15) Career recommendation model

In [16]:
def fit_career_model(df: pd.DataFrame) -> Dict[str, object]:
    X = df["resume_text"].tolist()
    y = df["category"].tolist()

    label_encoder = LabelEncoder()
    y_enc = label_encoder.fit_transform(y)

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y_enc,
        test_size=0.2,
        random_state=RANDOM_STATE,
        stratify=y_enc,
    )

    vectorizer = TfidfVectorizer(
        lowercase=True,
        strip_accents="unicode",
        stop_words="english",
        ngram_range=(1, 2),
        max_features=16000,
        min_df=2,
    )

    Xtr = vectorizer.fit_transform(X_train)
    Xte = vectorizer.transform(X_test)

    model = LogisticRegression(max_iter=1200, solver="lbfgs", random_state=RANDOM_STATE)
    model.fit(Xtr, y_train)

    prob = model.predict_proba(Xte)
    pred = prob.argmax(axis=1)

    k = min(3, len(label_encoder.classes_))
    metrics = {
        "accuracy": float(accuracy_score(y_test, pred)),
        "top_3_accuracy": float(top_k_accuracy_score(y_test, prob, k=k, labels=np.arange(len(label_encoder.classes_)))),
    }

    return {
        "vectorizer": vectorizer,
        "model": model,
        "label_encoder": label_encoder,
        "metrics": metrics,
    }

def recommend_careers(text: str, career_bundle: Dict[str, object], quality_result: Dict[str, object], top_k: int = 3) -> pd.DataFrame:
    X = career_bundle["vectorizer"].transform([text])
    prob = career_bundle["model"].predict_proba(X)[0]
    labels = career_bundle["label_encoder"].inverse_transform(np.arange(len(prob)))

    ranked = sorted(zip(labels, prob), key=lambda x: x[1], reverse=True)[:top_k]
    rows = []
    for label, p in ranked:
        rows.append(
            {
                "career_category": label,
                "career_probability": round(float(p), 4),
                "resume_quality_score": quality_result["score"],
                "resume_quality_rating": quality_result["rating"],
            }
        )
    return pd.DataFrame(rows)

## 16) Optional job-fit proxy model

In [17]:
def fit_optional_job_fit_model(ats_df: pd.DataFrame) -> Dict[str, object] | None:
    if ats_df is None or len(ats_df) == 0:
        return None

    fit_df = ats_df.dropna(subset=["resume_text", "ats_score"]).copy()
    fit_df = fit_df[fit_df["resume_text"].str.split().str.len() >= 40].copy()

    X_train, X_test, y_train, y_test = train_test_split(
        fit_df["resume_text"],
        fit_df["ats_score"],
        test_size=0.2,
        random_state=RANDOM_STATE,
    )

    vectorizer = TfidfVectorizer(
        lowercase=True,
        strip_accents="unicode",
        stop_words="english",
        ngram_range=(1, 2),
        max_features=10000,
        min_df=2,
    )

    Xtr = vectorizer.fit_transform(X_train.tolist())
    Xte = vectorizer.transform(X_test.tolist())

    bins = [-np.inf, 35, 60, np.inf]
    band_labels = ["low_fit", "medium_fit", "high_fit"]
    y_train_band = pd.cut(y_train, bins=bins, labels=band_labels)
    y_test_band = pd.cut(y_test, bins=bins, labels=band_labels)

    model = LogisticRegression(max_iter=1200, solver="lbfgs", random_state=RANDOM_STATE)
    model.fit(Xtr, y_train_band)

    pred = model.predict(Xte)
    prob = model.predict_proba(Xte)

    metrics = {
        "accuracy": float(accuracy_score(y_test_band, pred)),
        "top_2_accuracy": float(top_k_accuracy_score(y_test_band, prob, k=2, labels=model.classes_)),
    }

    return {
        "task": "optional_resume_job_fit_proxy",
        "vectorizer": vectorizer,
        "model": model,
        "metrics": metrics,
    }

## 17) Load datasets

This loads:
- ResumeAtlas
- ATS score dataset
- optional score-details dataset


In [18]:
career_df, ats_df, source_info = load_base_datasets()
optional_df = load_optional_score_details_dataset()

print("Base source info:", source_info)
print("Career rows:", len(career_df))
print("ATS rows:", len(ats_df))
print("Optional score-details rows:", len(optional_df))

display(career_df.head(2))
if len(optional_df):
    display(optional_df.head(2))

Resolving data files:   0%|          | 0/1031 [00:00<?, ?it/s]

Optional dataset not loaded: Expected object or value
Base source info: {'career': 'real', 'ats': 'real'}
Career rows: 13389
ATS rows: 5099
Optional score-details rows: 0


,category,resume_text
0,Accountant,education omba executive leadership university...
1,Accountant,howard gerrard accountant deyjobcom birmingham...


## 18) Train all models and save them

**This is the main save cell for backend use.**


In [19]:
quality_bundle = fit_quality_model(
    resumes=career_df["resume_text"].tolist(),
    optional_df=optional_df,
    max_samples=3000,
)

career_bundle = fit_career_model(career_df)
job_fit_bundle = fit_optional_job_fit_model(ats_df)

print("Intrinsic quality metrics:")
print(json.dumps(quality_bundle["metrics"], indent=2))

print("\nCareer metrics:")
print(json.dumps(career_bundle["metrics"], indent=2))

if job_fit_bundle is not None:
    print("\nOptional job-fit metrics:")
    print(json.dumps(job_fit_bundle["metrics"], indent=2))

joblib.dump(quality_bundle, ARTIFACT_DIR / "quality_bundle.joblib")
joblib.dump(career_bundle, ARTIFACT_DIR / "career_bundle.joblib")
if job_fit_bundle is not None:
    joblib.dump(job_fit_bundle, ARTIFACT_DIR / "job_fit_bundle.joblib")

training_summary = {
    "quality_metrics": quality_bundle["metrics"],
    "career_metrics": career_bundle["metrics"],
    "job_fit_metrics": None if job_fit_bundle is None else job_fit_bundle["metrics"],
    "optional_dataset_rows": int(len(optional_df)),
    "artifacts": [
        str(ARTIFACT_DIR / "quality_bundle.joblib"),
        str(ARTIFACT_DIR / "career_bundle.joblib"),
        str(ARTIFACT_DIR / "job_fit_bundle.joblib") if job_fit_bundle is not None else None,
    ],
}

with open(ARTIFACT_DIR / "training_summary.json", "w", encoding="utf-8") as f:
    json.dump(training_summary, f, indent=2)

print("\nSaved files:")
for name in ["quality_bundle.joblib", "career_bundle.joblib", "job_fit_bundle.joblib", "training_summary.json"]:
    p = ARTIFACT_DIR / name
    if p.exists():
        print("-", p)

Intrinsic quality metrics:
{
  "accuracy": 0.9058333333333334,
  "top_2_accuracy": 0.985,
  "train_rows": 12000,
  "synthetic_rows": 12000,
  "optional_rows_used": 0
}

Career metrics:
{
  "accuracy": 0.8058252427184466,
  "top_3_accuracy": 0.926437640029873
}

Optional job-fit metrics:
{
  "accuracy": 0.6019607843137255,
  "top_2_accuracy": 0.9166666666666666
}

Saved files:
- artifacts\quality_bundle.joblib
- artifacts\career_bundle.joblib
- artifacts\job_fit_bundle.joblib
- artifacts\training_summary.json


## 19) Choose a resume file for testing

In [41]:
# Put your own resume file path here.
RESUME_PATH ="Samman Khanal CV work.pdf"

# Examples:
# RESUME_PATH = "my_resume.pdf"
# RESUME_PATH = "my_resume.docx"

if RESUME_PATH and Path(RESUME_PATH).exists():
    resume_text = read_resume(RESUME_PATH)
    source_label = f"Loaded from file: {RESUME_PATH}"
else:
    resume_text = career_df.iloc[0]["resume_text"]
    source_label = "Using sample resume from ResumeAtlas"

print(source_label)
print()
print(resume_text[:2000] + ("..." if len(resume_text) > 2000 else ""))

Loaded from file: Samman Khanal CV work.pdf

Samman Khanal 
 
9807061637  
 
 Dharan, 
Sunsari 
samman.samman345@gmail.com 
linkedin.com/in/samman-khanal 
github.com/samman-khanal 
SUMMARY 
 
Motivated and detail-oriented Computer Science student with a strong foundation in software development, 
problem-solving, and collaborative team environments. Familiar in Java, Python, and web technologies, with 
academic experience in developing full-stack applications and working with databasesthrough assessments and 
personal projects. Strong communication and time management skills, with a proven ability to learn quickly and 
adapt to new challenges. Seeking an opportunity to contribute technical expertise and creativity to a forward-
thinking organization. 
EDUCATION 
BSc. (Hons) Computing 
Itahari International College 
London Metropolitan University 
2023 – present 
Dulari, Morang 
+2 in Computer Science 
Bishnu Memorial Secondary School 
PROJECTS 
2021 – 2023 
Dharan, Sunsari 
SammanTech 

## 20) Score the resume and recommend careers

In [42]:
quality_result = score_resume_quality(resume_text, quality_bundle)
career_result = recommend_careers(resume_text, career_bundle, quality_result, top_k=3)

print("Final resume quality score:", quality_result["score"])
print("Quality rating:", quality_result["rating"])
print("Raw score before penalty:", quality_result["raw_score"])
print("Penalty applied:", quality_result["penalty"])

print("\nQuality feedback:")
for item in quality_result["feedback"]:
    print("-", item)

print("\nKey structural signals:")
display(pd.DataFrame([quality_result["features"]]).T.rename(columns={0: "value"}))

print("\nRecommended careers:")
display(career_result)

print("\nQuality class probabilities:")
display(pd.DataFrame([quality_result["probabilities"]]))

Final resume quality score: 86.2
Quality rating: excellent
Raw score before penalty: 87.8
Penalty applied: 1.6

Quality feedback:
- Add more concise bullet points so recruiters can scan achievements faster.

Key structural signals:


,value
word_count,310.000000
line_count,67.000000
bullet_count,4.000000
section_coverage,6.000000
email_count,2.000000
phone_count,1.000000
year_mentions,11.000000
avg_line_length,4.313433
quantified_bullet_ratio,0.250000



Recommended careers:


,career_category,career_probability,resume_quality_score,resume_quality_rating
0,Python Developer,0.1275,86.2,excellent
1,Java Developer,0.1038,86.2,excellent
2,Web Designing,0.1027,86.2,excellent



Quality class probabilities:


,excellent,fair,good,poor
0,0.989269,0.000449,0.010282,5.809066e-11


## 21) Backend load test

In [43]:
loaded_quality_bundle = joblib.load(ARTIFACT_DIR / "quality_bundle.joblib")
loaded_career_bundle = joblib.load(ARTIFACT_DIR / "career_bundle.joblib")

print("Loaded quality bundle keys:", list(loaded_quality_bundle.keys()))
print("Loaded career bundle keys:", list(loaded_career_bundle.keys()))

if (ARTIFACT_DIR / "job_fit_bundle.joblib").exists():
    loaded_job_fit_bundle = joblib.load(ARTIFACT_DIR / "job_fit_bundle.joblib")
    print("Loaded job-fit bundle keys:", list(loaded_job_fit_bundle.keys()))

Loaded quality bundle keys: ['task', 'vectorizer', 'imputer', 'scaler', 'model', 'metrics', 'feature_percentiles', 'score_anchors', 'rating_thresholds']
Loaded career bundle keys: ['vectorizer', 'model', 'label_encoder', 'metrics']
Loaded job-fit bundle keys: ['task', 'vectorizer', 'model', 'metrics']
